# Determining Filtering Thresholds from Corpus Statistics

This notebook builds automatic filtering thresholds from a set of clean corpora. It assumes that corpora are already scored on the basis of multiple filters and that scores are saved in parquet files.

The main idea is:

1. Summarize each corpus using robust statistics for every filter.
2. Cluster corpora and pick a baseline cluster that looks internally consistent.
3. For each baseline corpus, compute document-level extreme quantiles such as the 99th percentile.
4. Measure how much those per-corpus thresholds disagree.
5. Use that disagreement to decide how strict or permissive the final corpus-level threshold should be.




In [1]:
import pandas as pd
import os
import numpy as np

#Fill in the following variables with the appropriate values for your use case
FOLDER = "<CORPUS_STATS_FOLDER>"
EXCLUDE_FILES = {"<File_1>", "<File_2>", "<File_3>"}

# List of filters to consider. The names should match the columns in the score files.
FILTER_COLS = [
    "NumbersFilter",
    "WordCountFilter",
    "SymbolsToWordsFilter",
    "UrlsFilter",
    "RepeatingTopNGramsFilter_n=2",
    "RepeatingTopNGramsFilter_n=3",
    "WhiteSpaceFilter",  # both
    "ParenthesesFilter",
    "BoilerPlateStringFilter",
    "RepeatedLinesFilter",  # left-tailed
    "RepeatedParagraphsFilter",  # left-tailed
    "RepeatedLinesByCharFilter",  # left-tailed
    "RepeatedParagraphsByCharFilter",  # left-tailed
    "NonAlphaNumericFilter",  # both
    "BulletsFilter",
    "LongWordFilter",
    "MeanWordLengthFilter",  # both
    "RepeatingDuplicateNGramsFilter_n=2",
    "RepeatingDuplicateNGramsFilter_n=3",
    "PunctuationFilter",  # both
    "EllipsisFilter",
]


LEFT_TAILED = {
    "RepeatedLinesFilter",
    "RepeatedParagraphsFilter",
    "RepeatedLinesByCharFilter",
    "RepeatedParagraphsByCharFilter",
}

BOTH_TAILED = {
    "WhiteSpaceFilter",
    "NonAlphaNumericFilter",
    "MeanWordLengthFilter",
    "PunctuationFilter",
}


## Summarize the Corpora

This cell builds a per-file summary table from a folder of Parquet files with corpus filter scores. For each corpus, it computes a small set of robust distribution statistics for selected columns, then stacks those summaries into one DataFrame. 

The resulting `stats_df` is a metadata table of feature distributions per file. Each row summarizes one corpus’s data distribution for the selected columns using:

- a robust center: `median`
- a robust scale: `log_spread`
- an upper-tail extremeness measure: `log_tail`
- optionally a lower-tail extremeness measure via negation: `log_tail_flip`

These features are chosen to describe the typical value, the spread of data, and tail heaviness in a way that is robust to outliers and can be compared across different corpora. We will use them to group similar corpora together and choose the most suitable group for threshold determination.

In [2]:
EPS = 1e-9 #prevents log(0) in tail calculations
SHIFT = 1e-6 #ensures positive values for log_tail calculations, even if the distribution has a very long left tail
LOW_SHIFT_Q = 0.001 #use the 0.1% for log_tail to avoid extreme outliers dominating the tail spread calculation

all_stats = []

for file in os.listdir(FOLDER):
    if not file.endswith(".parquet") or file in EXCLUDE_FILES:
        continue

    df = pd.read_parquet(os.path.join(FOLDER, file))
    stats = {}

    for c in FILTER_COLS:
        need_flip = (c in LEFT_TAILED) or (c in BOTH_TAILED)

        stats[f"{c}__median"] = np.nan
        stats[f"{c}__log_spread"] = np.nan
        stats[f"{c}__log_tail"] = np.nan
        if need_flip:
            stats[f"{c}__log_tail_flip"] = np.nan

        if c not in df.columns:
            continue

        x = df[c].dropna().to_numpy()
        if x.size == 0:
            continue

        # --- median + spread on priginal only ---
        q25, q50, q75 = np.quantile(x, [0.25, 0.50, 0.75])
        stats[f"{c}__median"] = q50
        stats[f"{c}__log_spread"] = np.log1p(max(q75 - q25, 0.0))

        def log_tail_right(v: np.ndarray) -> float:
            vlo = np.quantile(v, LOW_SHIFT_Q)
            z = v - vlo + SHIFT
            z50, z99 = np.quantile(z, [0.50, 0.99])
            return float(np.log(z99 + EPS) - np.log(z50 + EPS))

        # --- log_tail on original ---
        stats[f"{c}__log_tail"] = log_tail_right(x)

        # --- log_tail on flip ---
        if need_flip:
            stats[f"{c}__log_tail_flip"] = log_tail_right(-x)
            

    row = pd.Series(stats, name=os.path.splitext(file)[0])
    all_stats.append(row)

stats_df = pd.DataFrame(all_stats)
print(stats_df.head())


FileNotFoundError: [Errno 2] No such file or directory: '<CORPUS_STATS_FOLDER>'

## Cluster Corpora to Find a Baseline Group

Now we cluster the corpus summaries in `stats_df`. Some corpora may be noisy, domain-specific, or simply unlike the rest,  so we first identify a baseline cluster of corpora that we can trust.

Implementation choices:
- `RobustScaler` to match robust stats.
- `HDBSCAN` because it can find clusters of uneven density and label unusual corpora as noise (`-1`).

The output columns:
- `hdb_label`: cluster assignment
- `hdb_outlier`: outlier score

In [ ]:
from sklearn.preprocessing import RobustScaler
import hdbscan

X = stats_df.fillna(stats_df.median())
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

#sample parameters
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    min_samples=3,     
    metric="euclidean",
    cluster_selection_method="eom" 
)

labels = clusterer.fit_predict(X_scaled)                           
outlier = clusterer.outlier_scores_                

stats_df = stats_df.copy()
stats_df["hdb_label"] = labels
stats_df["hdb_outlier"] = outlier

print("Cluster sizes (HDBSCAN):")
print(stats_df["hdb_label"].value_counts().sort_index())

for lab in sorted(stats_df["hdb_label"].unique()):
    members = stats_df.index[stats_df["hdb_label"] == lab].tolist()
    if lab == -1:
        print(f"\nNoise / outliers (-1): {members}")
    else:
        print(f"\nCluster {lab}: {members}")


## Visual Check of the Clustering

This PCA plot helps answer whether the corpora form interpretable groups and which features drive the separation the most.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler 
from sklearn.decomposition import PCA

# --- data ---
cols = [c for c in stats_df.columns if c not in ["hdb_outlier", "hdb_label"]]
X = stats_df[cols].fillna(stats_df[cols].median())

# --- scale ---
scaler = RobustScaler()                 
pca_matrix = X
X_scaled = scaler.fit_transform(pca_matrix)

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- plot ---
labels = stats_df.index
plt.figure(figsize=(6, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=stats_df["hdb_label"], cmap="tab10")
for i, name in enumerate(labels):
    plt.text(X_pca[i, 0], X_pca[i, 1], name, fontsize=8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


# --- Loadings ---
loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_matrix.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

print("Explained variance ratio:", pca.explained_variance_ratio_)

def plot_loadings(loadings_df, pc="PC1", top_n=20):
    vals = loadings_df[pc].sort_values(key=np.abs, ascending=False).head(top_n)
    plt.figure(figsize=(10, 4))
    vals.sort_values().plot(kind="bar")
    plt.axhline(0, linewidth=1)
    plt.title(f"{pc} loadings (top {top_n} by absolute value)")
    plt.ylabel("Loading")
    plt.show()

plot_loadings(loadings, "PC1", top_n=20)
plot_loadings(loadings, "PC2", top_n=20)



## Choose the Baseline Corpora

Here we manually choose one HDBSCAN cluster as the baseline set. This is the set of corpora we trust for threshold induction.


In [ ]:
#Choose your baseline cluster and fill in the label here. 
BASELINE_CLUSTER_LABEL = "<BASELINE_CLUSTER_LABEL>"

BASELINE_IDS = stats_df.index[stats_df["hdb_label"] == BASELINE_CLUSTER_LABEL].tolist()
print("Baseline corpora:", BASELINE_IDS)
print("N baseline:", len(BASELINE_IDS))


## Determine Final Thresholds


### 1: `collect_thresholds`
For every baseline corpus and every filter, we compute a document-level extreme quantile.

With `q_doc = 0.99`:
- regular right-tailed filters use the 99th percentile
- left-tailed filters use the 1st percentile
- both-tailed filters collect both sides

After this step, each filter has a list of candidate thresholds, one candidate per baseline corpus.


In [ ]:
def collect_thresholds(folder, baseline_ids, cols, left_tailed, both_tailed, q_doc=0.99):
    """Per clean corpus: compute q99 (or q01) thresholds for each filter."""
    hi = {c: [] for c in cols}
    lo = {c: [] for c in cols}

    for file in os.listdir(folder):
        if not file.endswith(".parquet") or file in EXCLUDE_FILES:
            continue
        corpus_id = os.path.splitext(file)[0]
        if corpus_id not in baseline_ids:
            continue

        df = pd.read_parquet(os.path.join(folder, file), columns=cols)

        for c in cols:
            x = pd.to_numeric(df[c]).dropna().to_numpy()
            if x.size == 0:
                continue

            if c in left_tailed:
                lo[c].append(float(np.quantile(x, 1.0 - q_doc)))
            elif c in both_tailed:
                lo[c].append(float(np.quantile(x, 1.0 - q_doc)))
                hi[c].append(float(np.quantile(x, q_doc)))
            else:
                hi[c].append(float(np.quantile(x, q_doc)))

    return hi, lo

### 2: `variability_score`

We then measure how much those candidate thresholds disagree across baseline corpora. We define variability between corpora as
$$
\text{variability} = \frac{\text{MAD}}{|\text{median}| + \epsilon}
$$
where MAD is the median absolute deviation.
If the variability is high for a particular filter, that means that even for clean corpora, the filter scores are very corpus-dependent, so the final threshold should be more permissive. If, however, the variability is low, we can tighten the thresholds.


### 3: `choose_permissiveness`
This converts variability into a per-filter `q_corp` that will determine threshold permissiveness. Stable filters get a lower `q_corp`, which means stricter aggregation.
More variable filters get a higher `q_corp` with more permissive aggregation. We set lower and upper boundaries `qcorp_min` and `qcorp_max`. 


In [ ]:
def variability_score(vals):
    """Relative disagreement across clean corpora: MAD / |median|."""
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.inf
    med = np.median(vals)
    mad = np.median(np.abs(vals - med))
    return mad / (abs(med) + 1e-9)

def choose_permissiveness(cols, left_tailed, both_tailed, hi, lo,
                          qcorp_min=0.80, qcorp_max=0.98):
    """More disagreement across clean corpora => more permissive."""
    var = {}
    for c in cols:
        vals = []
        if c in left_tailed or c in both_tailed:
            vals += lo[c]
        if c not in left_tailed:
            vals += hi[c]
        var[c] = variability_score(np.asarray(vals, float))

    v = np.asarray(list(var.values()), float)
    v_min, v_max = np.min(v), np.max(v)

    q_corp = {}
    for c, vs in var.items():
        if not np.isfinite(vs) or v_max == v_min:
            q_corp[c] = (qcorp_min + qcorp_max) / 2.0
        else:
            t = (vs - v_min) / (v_max - v_min)
            q_corp[c] = qcorp_min + t * (qcorp_max - qcorp_min)

    return q_corp 

### 4: `aggregate_thresholds`
Finally, we aggregate the per-corpus candidate thresholds into one final threshold per filter.

- right-tailed filter: final upper threshold is the `q_corp` quantile of per-corpus upper thresholds
- left-tailed filter: final lower threshold is the `(1 - q_corp)` quantile of per-corpus lower thresholds
- both-tailed filter: do both independently

This produces:
- `T_hi`: final upper thresholds
- `T_lo`: final lower thresholds

In [ ]:
def aggregate_thresholds(cols, left_tailed, both_tailed, hi, lo, q_corp):
    """Aggregate per-corpus thresholds into one threshold per filter."""
    hi_final, lo_final = {}, {}

    for c in cols:
        qc = q_corp[c]

        if c in left_tailed:
            vals = np.asarray(lo[c], float)
            lo_final[c] = float(np.quantile(vals, 1.0 - qc)) 

        elif c in both_tailed:
            vals_lo = np.asarray(lo[c], float)
            vals_hi = np.asarray(hi[c], float)
            lo_final[c] = float(np.quantile(vals_lo, 1.0 - qc)) 
            hi_final[c] = float(np.quantile(vals_hi, qc)) 

        else:
            vals = np.asarray(hi[c], float)
            hi_final[c] = float(np.quantile(vals, qc)) 

    return pd.Series(hi_final, name="T_hi").sort_index(), pd.Series(lo_final, name="T_lo").sort_index()

## Run the Threshold Pipeline

This cell executes the full procedure:

1. `collect_thresholds(...)` gathers per-baseline-corpus document-level extremes
2. `choose_permissiveness(...)` computes adaptive `q_corp` for each filter
3. `aggregate_thresholds(...)` turns those into final thresholds

Parameter meanings:
- `q_doc=0.99`: define document-level extremes at the 99th / 1st percentile
- `qcorp_min=0.80`: minimum cross-corpus permissiveness for very stable filters
- `qcorp_max=0.98`: maximum cross-corpus permissiveness for unstable filters

A good way to read the output:
- `q_corp per filter` tells you which filters are considered stable vs unstable
- `Upper thresholds` and `Lower thresholds` are the final values you will actually apply

In [ ]:
q_doc=0.99
qcorp_min=0.80
qcorp_max=0.98
hi, lo = collect_thresholds(FOLDER, BASELINE_IDS, FILTER_COLS, LEFT_TAILED, BOTH_TAILED, q_doc)
q_corp = choose_permissiveness(FILTER_COLS, LEFT_TAILED, BOTH_TAILED, hi, lo, qcorp_min, qcorp_max)
T_hi, T_lo = aggregate_thresholds(FILTER_COLS, LEFT_TAILED, BOTH_TAILED, hi, lo, q_corp)

print("q_corp per filter:")
print(pd.Series(q_corp))

print("\nUpper thresholds:")
print(T_hi.dropna())

print("\nLower thresholds:")
print(T_lo.dropna())


## Save the Learned Thresholds

We can save the thresholds as JSON so they can be reused in a filtering pipeline without rerunning the notebook.

The saved file contains:
- the baseline corpora used,
- filter metadata,
- tail-direction metadata,
- document-level and corpus-level quantile settings,
- per-filter `q_corp`,
- final upper and lower thresholds.

In [ ]:
import json

OUTPUT_DIR = "<OUTPUT_DIRECTORY>"
OUTPUT_FILE = "<OUTPUT_FILE>"

def save_thresholds(baseline_ids, filter_cols, left_tailed, both_tailed, q_doc,
                    qcorp_min, qcorp_max, T_hi, T_lo, q_corp):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    threshold_data = {
        "baseline_corpora": list(baseline_ids),
        "filters": list(filter_cols),
        "left_tailed": list(left_tailed),
        "both_tailed": list(both_tailed),
        "q_doc": q_doc,
        "qcorp_min": qcorp_min,
        "qcorp_max": qcorp_max,
        "q_corp_per_filter": {k: float(v) for k, v in q_corp.items()},
        "upper_thresholds": {k: float(v) for k, v in T_hi.dropna().items()},
        "lower_thresholds": {k: float(v) for k, v in T_lo.dropna().items()},
    }

    # --- save file ---
    output_file = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(threshold_data, f, indent=4)

save_thresholds(BASELINE_IDS, FILTER_COLS, LEFT_TAILED, BOTH_TAILED, q_doc,
                qcorp_min, qcorp_max, T_hi, T_lo, q_corp)
